# Embedding Fails Gallery

**AI-2 Session 2.1 Companion — Self-Paced**

---

In `how_models_think.ipynb`, you saw that embedding models and LLMs process text in fundamentally different ways. Embeddings **compress** text into fixed-length vectors — fast and useful, but lossy. Some things that are obvious to Claude — negation, word order, tone — get flattened or lost entirely in that compression.

This notebook exposes **5 categories of failure** where embeddings give misleading similarity scores. Each section presents curated text pairs, computes cosine similarity, and asks you to interpret the result.

**Why this matters:** These failure modes are exactly why naive RAG retrieval breaks down. When you build retrieval pipelines in Session 3.1, you will see these problems surface in real search results. Understanding them now gives you the vocabulary to diagnose retrieval failures later.

| Category | What Goes Wrong |
|----------|----------------|
| 1. Polysemy | Same word, different meaning — embeddings conflate them |
| 2. Negation | "is" vs "is not" looks nearly identical to the model |
| 3. Word Order | Swapping subject and object changes meaning, not always the vector |
| 4. Length Mismatch | Short queries vs long passages produce skewed scores |
| 5. Register / Jargon | Formal and informal language for the same concept diverge |

## Setup

We import the same `embed.py` module used in the walk-along notebook. Make sure you are running this notebook from the repo root (or that `AI-2/src` is on your Python path).

In [ ]:
import sys
import os

# Ensure the src directory is importable
# This handles running from the notebooks/ directory or the AI-2/ root
src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if src_path not in sys.path:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from dotenv import load_dotenv
load_dotenv()

from src.s2_embeddings.embed import get_embedding, cosine_similarity

# Shared score collector for the summary radar chart
all_scores = {}

print("Setup complete. Ready to explore embedding failures.")

## Helper Function

The `compare_pair` function embeds two texts, computes cosine similarity, and prints a formatted result with a visual similarity bar. We will use this throughout the notebook.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def compare_pair(text_a: str, text_b: str, label: str = "", model: str = "voyage-3-lite", expected: str = None) -> float:
    """Embed two texts, compute cosine similarity, and display a visual gauge.
    
    Args:
        text_a: First text to compare.
        text_b: Second text to compare.
        label: Optional label for this comparison.
        model: Voyage AI model to use (default: voyage-3-lite).
        expected: "similar" (texts mean the same), "different" (texts mean different things),
                  or None (no judgment shown).
    
    Returns:
        The cosine similarity score.
    """
    vec_a = get_embedding(text_a, model=model)
    vec_b = get_embedding(text_b, model=model)
    score = cosine_similarity(vec_a, vec_b)
    
    # --- Matplotlib gradient gauge ---
    fig, ax = plt.subplots(figsize=(6, 1.2))
    
    # Build a horizontal gradient bar (green -> yellow -> red)
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    cmap = mcolors.LinearSegmentedColormap.from_list("sim", ["#2ecc71", "#f1c40f", "#e74c3c"])
    ax.imshow(gradient, aspect="auto", cmap=cmap, extent=[0, 1, 0, 1])
    
    # Score marker
    ax.axvline(x=score, color="black", linewidth=2.5, ymin=0.0, ymax=1.0)
    
    # Retrieval threshold reference line
    ax.axvline(x=0.70, color="white", linewidth=1.2, linestyle="--", alpha=0.9)
    ax.text(0.70, 1.12, "retrieval\nthreshold", fontsize=7, color="gray",
            ha="center", va="bottom", transform=ax.get_xaxis_transform())
    
    # Score display
    ax.text(score, -0.25, f"{score:.4f}", fontsize=14, fontweight="bold",
            ha="center", va="top", transform=ax.get_xaxis_transform())
    
    # Determine if the score is misleading
    indicator = ""
    if expected == "similar":
        indicator = " \u2705" if score >= 0.70 else " \u274c misleading"
    elif expected == "different":
        indicator = " \u274c misleading" if score >= 0.70 else " \u2705"
    
    # Title with label and indicator
    title = label if label else "Comparison"
    ax.set_title(f"{title}{indicator}", fontsize=9, fontweight="bold", loc="left", pad=8)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.tick_params(axis="x", labelsize=7)
    
    # Text labels below the figure
    fig.text(0.08, -0.02, f'A: "{text_a[:80]}{"..." if len(text_a) > 80 else ""}"',
             fontsize=7, color="#555555", va="top")
    fig.text(0.08, -0.12, f'B: "{text_b[:80]}{"..." if len(text_b) > 80 else ""}"',
             fontsize=7, color="#555555", va="top")
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    plt.close()
    
    return score

print("compare_pair() ready — with visual gauge.")

---

## Failure Category 1: Polysemy

**The problem:** A single word can have completely different meanings depending on context. Embedding models compress the entire sentence into one vector, and sometimes the shared word pulls the vectors closer together than they should be.

**Why it matters for RAG:** If your knowledge base contains documents about "models" (ML performance benchmarks) and a user asks about "runway models," polysemy can cause the retrieval system to return irrelevant machine learning documents. The same applies to "drivers" (software vs automotive), "python" (language vs snake), and countless other terms.

In [ ]:
score_1 = compare_pair(
    "The model performed well on the test set",
    "The model performed well on the runway",
    label="Polysemy: 'model' (ML model vs fashion model)",
    expected="different"
)

score_2 = compare_pair(
    "The driver crashed after the update",
    "The driver crashed after the turn",
    label="Polysemy: 'driver' (software vs car)",
    expected="different"
)

score_3 = compare_pair(
    "Python is great for processing large volumes of data",
    "The python is great for consuming large volumes of prey",
    label="Polysemy: 'python' (language vs snake)",
    expected="different"
)

# Baseline: actually different meaning AND different words
score_baseline = compare_pair(
    "The model performed well on the test set",
    "The financial institution granted the credit application",
    label="Baseline: completely unrelated sentences",
    expected="different"
)

all_scores["Polysemy"] = {"scores": [score_1, score_2, score_3], "type": "should_differ"}

print("\n" + "=" * 60)
print("OBSERVATION: The polysemy pairs may score HIGHER than you'd expect.")
print("When sentences share the same structure AND a key word, the model")
print("conflates them even though the domains are completely different.")
print()
print("Compare the polysemy scores to the baseline (unrelated sentences).")
print("If polysemy pairs score close to or above the baseline, the model")
print("is being fooled by surface similarity.")
print("=" * 60)

---

## Failure Category 2: Negation

**The problem:** Adding "not" to a sentence completely reverses its meaning. But the vast majority of the tokens remain the same, so the embedding vectors stay dangerously close together.

**Why it matters for RAG:** Imagine a policy document that says "Employees are eligible for overtime pay." A user asks: "Are employees NOT eligible for overtime?" The retrieval system finds the right document -- but the embedding similarity is high for both the affirmed and negated version. The system cannot distinguish compliance from violation.

In [ ]:
# Pair 1: Project completion status
score_1 = compare_pair(
    "The project was completed on time",
    "The project was NOT completed on time",
    label="Negation: completed vs NOT completed",
    expected="different"
)

# Pair 2: Employee eligibility
score_2 = compare_pair(
    "Employees are eligible for benefits after 90 days",
    "Employees are not eligible for benefits after 90 days",
    label="Negation: eligible vs not eligible",
    expected="different"
)

# Pair 3: A more subtle negation
score_3 = compare_pair(
    "The system passed all security audits",
    "The system failed to pass security audits",
    label="Negation: passed vs failed to pass",
    expected="different"
)

all_scores["Negation"] = {"scores": [score_1, score_2, score_3], "type": "should_differ"}

print("\n" + "=" * 60)
print("OBSERVATION: These pairs have OPPOSITE meanings.")
print(f"Yet the similarity scores are: {score_1:.4f}, {score_2:.4f}, {score_3:.4f}")
print("Scores above 0.90 are especially dangerous -- the model")
print("essentially cannot tell these apart.")
print("=" * 60)

### Try It Yourself: The Negation Flipper

Type **any statement** into the box below and press Enter. The notebook will automatically negate it and show you the similarity score. Try different topics, lengths, and complexity levels.

**The pattern you'll see:** No matter what you type, the similarity stays dangerously high (typically 0.85+). The embedding model fundamentally cannot distinguish a statement from its negation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Interactive Negation Flipper ---
print("Type any statement below. The notebook will negate it and show")
print("how similar the embedding model thinks the original and negation are.\n")

text_input = widgets.Text(
    value="The project was delivered on time and under budget",
    placeholder="Type a statement...",
    description="Statement:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "80px"},
)

output_area = widgets.Output()

negation_log = []

def on_submit(change):
    original = text_input.value.strip()
    if not original:
        return
    negated = f"It is not true that {original[0].lower()}{original[1:]}"
    
    vec_a = get_embedding(original)
    vec_b = get_embedding(negated)
    score = cosine_similarity(vec_a, vec_b)
    negation_log.append({"original": original, "negated": negated, "score": score})
    
    with output_area:
        clear_output(wait=True)
        
        # Gauge-style display
        fig, ax = plt.subplots(figsize=(8, 0.6))
        bar_color = "crimson" if score > 0.85 else "orange" if score > 0.7 else "green"
        ax.barh(0, score, height=0.4, color=bar_color, alpha=0.8)
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        ax.axvline(x=0.85, color="red", linestyle="--", linewidth=1, alpha=0.7)
        ax.text(0.85, 0.35, "danger zone", fontsize=8, color="red", ha="center")
        ax.set_title(f"Similarity: {score:.4f}", fontsize=14, fontweight="bold")
        ax.set_xlabel("Cosine Similarity")
        plt.tight_layout()
        plt.show()
        plt.close()
        
        print(f'  Original: "{original}"')
        print(f'  Negated:  "{negated}"')
        print(f"  Score:    {score:.4f}")
        print()
        
        if len(negation_log) > 1:
            scores = [entry["score"] for entry in negation_log]
            print(f"  --- Running stats ({len(negation_log)} attempts) ---")
            print(f"  Min: {min(scores):.4f}  Max: {max(scores):.4f}  Avg: {np.mean(scores):.4f}")
            print(f"  Every single one is dangerously similar to its negation.")

text_input.observe(on_submit, names="value")

display(text_input, output_area)

# Trigger initial example
on_submit(None)

---

## Failure Category 3: Word Order

**The problem:** Rearranging who does what to whom changes meaning, but if the same words are present, the embedding may not reflect the change. Embedding models are not parsers -- they do not always encode syntactic structure faithfully.

**Why it matters for RAG:** In HR or legal contexts, "the manager approved the employee's request" and "the employee approved the manager's request" describe completely different authority flows. A retrieval system that treats them as equivalent will surface the wrong policy.

In [ ]:
# Pair 1: Who approved whom
score_1 = compare_pair(
    "The manager approved the employee's request",
    "The employee approved the manager's request",
    label="Word Order: manager approves employee vs employee approves manager",
    expected="different"
)

# Pair 2: Who chases whom
score_2 = compare_pair(
    "Dogs chase cats",
    "Cats chase dogs",
    label="Word Order: dogs chase cats vs cats chase dogs",
    expected="different"
)

# Pair 3: Directional relationship
score_3 = compare_pair(
    "The vendor sent the invoice to the client",
    "The client sent the invoice to the vendor",
    label="Word Order: vendor invoices client vs client invoices vendor",
    expected="different"
)

all_scores["Word Order"] = {"scores": [score_1, score_2, score_3], "type": "should_differ"}

print("\n" + "=" * 60)
print("OBSERVATION: Same words, reversed roles, different meaning.")
print(f"Similarity scores: {score_1:.4f}, {score_2:.4f}, {score_3:.4f}")
print("High scores here mean the model cannot distinguish")
print("who is the actor and who is the recipient.")
print("=" * 60)

---

## Failure Category 4: Length Mismatch

**The problem:** Embedding models compress text into a fixed-length vector regardless of input length. A 5-word query and a 500-word document both become the same size vector. This compression is inherently lossy and can produce misleading similarity scores -- long passages that contain an answer may score *lower* than shorter, less relevant passages.

**Why it matters for RAG:** This is one of the most common failure modes in real retrieval systems. Users type short questions. Documents are long. The length asymmetry systematically distorts similarity scores. Chunking strategy (Session 2.2) is partly a response to this problem.

In [ ]:
# The short query
query = "What is the remote work policy?"

# A long passage that CONTAINS the answer (buried in context)
long_relevant = (
    "Northbrook Partners Employee Handbook - Section 7: Workplace Policies. "
    "This section covers all policies related to the physical and virtual workplace. "
    "Employees are expected to maintain professional standards whether working on-site "
    "or remotely. The dress code policy applies to all in-person meetings and client "
    "interactions. Office hours are 9:00 AM to 5:00 PM local time, with flexibility "
    "granted on a case-by-case basis. Regarding remote work arrangements, employees "
    "who have completed their 90-day probationary period may request up to 3 days per "
    "week of remote work with manager approval. Remote workers must maintain a dedicated "
    "workspace, be available during core hours (10 AM - 3 PM), and attend all mandatory "
    "in-person meetings. Equipment stipends of $500 are available for home office setup. "
    "All remote work arrangements are subject to quarterly review and may be revoked if "
    "performance metrics are not met. Additional policies regarding travel reimbursement, "
    "conference attendance, and professional development are covered in Section 8."
)

# A long passage on a DIFFERENT topic
long_irrelevant = (
    "Northbrook Partners Employee Handbook - Section 12: Benefits and Compensation. "
    "This section outlines the comprehensive benefits package available to all full-time "
    "employees. Health insurance coverage begins on the first day of the month following "
    "your start date. Northbrook Partners offers three tiers of medical coverage: Bronze, "
    "Silver, and Gold. Dental and vision coverage is included in all tiers. Life insurance "
    "at 2x annual salary is provided at no cost to the employee. The 401(k) retirement "
    "plan includes employer matching up to 4% of gross salary. Employees are immediately "
    "vested in employer contributions. Paid time off accrues at 15 days per year for the "
    "first 3 years, increasing to 20 days after 3 years and 25 days after 7 years. "
    "Sick leave is tracked separately at 10 days per year. Parental leave of 12 weeks "
    "is available to all new parents regardless of gender. Additional benefits including "
    "tuition reimbursement and wellness programs are described in the benefits portal."
)

# A short passage that IS relevant
short_relevant = "Remote work is allowed 3 days per week after the probationary period."

print("Query:", query)
print(f"Query length: {len(query.split())} words")
print(f"Long relevant passage: {len(long_relevant.split())} words")
print(f"Long irrelevant passage: {len(long_irrelevant.split())} words")
print(f"Short relevant passage: {len(short_relevant.split())} words")
print()

score_long_relevant = compare_pair(
    query, long_relevant,
    label="Short query vs LONG passage (contains the answer)",
    expected="similar"
)

score_long_irrelevant = compare_pair(
    query, long_irrelevant,
    label="Short query vs LONG passage (different topic entirely)",
    expected="different"
)

score_short_relevant = compare_pair(
    query, short_relevant,
    label="Short query vs SHORT passage (contains the answer)",
    expected="similar"
)

all_scores["Length\nMismatch"] = {
    "scores": [score_long_relevant, score_short_relevant],
    "type": "should_match",
    "gap": abs(score_short_relevant - score_long_relevant)
}

print("\n" + "=" * 60)
print("OBSERVATION: Compare the three scores.")
print(f"  Long relevant:    {score_long_relevant:.4f}")
print(f"  Long irrelevant:  {score_long_irrelevant:.4f}")
print(f"  Short relevant:   {score_short_relevant:.4f}")
print()
print("The short relevant passage likely scores HIGHEST because")
print("it is closest in length to the query. The long relevant")
print("passage -- which actually contains the answer -- may score")
print("lower because the answer is diluted by surrounding text.")
print("\nThis is why CHUNKING matters. Session 2.2 addresses this.")
print("=" * 60)

---

## Failure Category 5: Register and Jargon

**The problem:** The same concept expressed in formal/technical language vs casual/everyday language can produce very different embeddings. Domain-specific jargon and colloquial equivalents may land far apart in embedding space even though they mean the same thing.

**Why it matters for RAG:** If your knowledge base is written in formal HR or legal language, but users ask questions in casual everyday language, the retrieval system may fail to match them. This is a gap between how documents are written and how people search.

In [ ]:
# Pair 1: Medical jargon vs plain language
score_1 = compare_pair(
    "The patient presented with dyspnea and tachycardia",
    "The patient couldn't breathe and their heart was racing",
    label="Register: medical jargon vs plain language",
    expected="similar"
)

# Pair 2: HR formal policy vs casual Slack message
score_2 = compare_pair(
    "The paid time off policy has been updated effective Q2",
    "Hey, vacation days changed starting next quarter",
    label="Register: formal HR vs casual Slack",
    expected="similar"
)

# Pair 3: Legal vs everyday
score_3 = compare_pair(
    "The party of the first part shall indemnify and hold harmless the party of the second part",
    "If something goes wrong, the company will cover the costs and protect you",
    label="Register: legal boilerplate vs plain explanation",
    expected="similar"
)

# Pair 4: Technical acronym vs expanded form
score_4 = compare_pair(
    "Implement RBAC with OIDC SSO integration",
    "Set up role-based access control with single sign-on",
    label="Register: acronyms vs expanded terms",
    expected="similar"
)

all_scores["Register"] = {"scores": [score_1, score_2, score_3, score_4], "type": "should_match"}

print("\n" + "=" * 60)
print("OBSERVATION: These pairs mean the SAME thing in different registers.")
print(f"Scores: {score_1:.4f}, {score_2:.4f}, {score_3:.4f}, {score_4:.4f}")
print("Scores below 0.80 indicate the model struggles to bridge")
print("the gap between formal and informal language.")
print("\nThis matters when your documents are written one way")
print("but your users search in a completely different register.")
print("=" * 60)

---

## The Challenge: Build Your Own Fails

Your turn. For each category below, write a pair of sentences that you believe the embedding model will get **wrong**. The `test_my_fail()` function will tell you if the model was fooled.

**Goal: Can you fool the model on 3 out of 5 categories?**

Tips:
- For Negation/Word Order: keep the structure identical, flip one key element
- For Polysemy: use the same word in completely different domains
- For Register: say the same thing in formal vs casual language
- For Length: write a short question and a long passage that contains the answer

In [ ]:
def test_my_fail(category: str, text_a: str, text_b: str, expected: str = "different") -> bool:
    """Test if the embedding model gets your pair wrong.
    
    Args:
        category: Which failure mode you're testing
        text_a: First text
        text_b: Second text  
        expected: "different" (texts mean different things) or "similar" (same meaning)
    
    Returns:
        True if the model was fooled, False if it got it right
    """
    score = compare_pair(text_a, text_b, label=f"Your test: {category}", expected=expected)
    
    # Threshold: 0.70 is a typical retrieval cutoff
    if expected == "different" and score > 0.70:
        print(f"  YOU FOOLED IT! Score {score:.4f} > 0.70 -- model thinks these are similar")
        return True
    elif expected == "similar" and score < 0.50:
        print(f"  YOU FOOLED IT! Score {score:.4f} < 0.50 -- model thinks these are different")
        return True
    else:
        print(f"  Model got it right this time. Try a different pair!")
        return False

# --- YOUR CHALLENGES ---
# Uncomment each block and fill in your sentences

# Challenge 1: Polysemy
# result_1 = test_my_fail("Polysemy",
#     "YOUR SENTENCE A",
#     "YOUR SENTENCE B",
#     expected="different"
# )

# Challenge 2: Negation
# result_2 = test_my_fail("Negation",
#     "YOUR SENTENCE A",
#     "YOUR SENTENCE B (negated version)",
#     expected="different"
# )

# Challenge 3: Word Order
# result_3 = test_my_fail("Word Order",
#     "YOUR SENTENCE A",
#     "YOUR SENTENCE B (roles swapped)",
#     expected="different"
# )

# Challenge 4: Register
# result_4 = test_my_fail("Register",
#     "YOUR FORMAL SENTENCE",
#     "YOUR CASUAL SENTENCE (same meaning)",
#     expected="similar"
# )

# Challenge 5: Length Mismatch
# result_5 = test_my_fail("Length Mismatch",
#     "A short question",
#     "A long paragraph that contains the answer buried in lots of other text...",
#     expected="similar"
# )

# --- SCOREBOARD ---
# Uncomment after filling in all 5:
# wins = sum([result_1, result_2, result_3, result_4, result_5])
# print(f"\n{'='*40}")
# print(f"  SCORE: {wins}/5 -- {'You beat the model!' if wins >= 3 else 'The model wins this round.'}")
# print(f"{'='*40}")

---

## The Failure Profile

How broken is each category? This radar chart shows the "failure severity" -- how misleading the similarity scores are for each type of embedding failure.

- **Negation & Word Order** spike high because the model produces DANGEROUSLY similar scores for texts with opposite meanings
- **Register** shows a different problem: low scores for texts that SHOULD be similar
- **Polysemy** and **Length Mismatch** fall in between

In [ ]:
import plotly.graph_objects as go

# Compute failure severity per category
# For "should_differ" categories: severity = mean_similarity * 10 (high sim when should differ = bad)
# For "should_match" categories: severity = (1 - mean_similarity) * 10 (low sim when should match = bad)
# For "Length Mismatch": severity based on the gap between short and long relevant scores

categories = ["Negation", "Word Order", "Polysemy", "Length\nMismatch", "Register"]
severities = []

for cat in categories:
    if cat not in all_scores:
        severities.append(0)
        continue
    entry = all_scores[cat]
    mean_score = np.mean(entry["scores"])
    
    if entry["type"] == "should_differ":
        # High similarity for texts that should be different = high severity
        severities.append(mean_score * 10)
    elif entry["type"] == "should_match":
        if "gap" in entry:
            # Length mismatch: severity based on how much the long passage loses vs short
            severities.append(entry["gap"] * 10 + (1 - mean_score) * 5)
        else:
            # Register: low similarity for texts that should match = high severity
            severities.append((1 - mean_score) * 10)

# Close the radar polygon
categories_closed = categories + [categories[0]]
severities_closed = severities + [severities[0]]
ideal_closed = [0] * len(categories_closed)

fig = go.Figure()

# Actual failure severity (red filled)
fig.add_trace(go.Scatterpolar(
    r=severities_closed,
    theta=categories_closed,
    fill="toself",
    fillcolor="rgba(231, 76, 60, 0.25)",
    line=dict(color="rgba(231, 76, 60, 0.9)", width=2.5),
    name="Failure Severity",
    marker=dict(size=8, color="#e74c3c"),
))

# Ideal baseline (green dashed)
fig.add_trace(go.Scatterpolar(
    r=ideal_closed,
    theta=categories_closed,
    fill="toself",
    fillcolor="rgba(46, 204, 113, 0.1)",
    line=dict(color="rgba(46, 204, 113, 0.7)", width=1.5, dash="dash"),
    name="Ideal (no failure)",
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10],
            tickvals=[2, 4, 6, 8, 10],
            ticktext=["2", "4", "6", "8", "10"],
            gridcolor="rgba(0,0,0,0.1)",
        ),
        angularaxis=dict(
            gridcolor="rgba(0,0,0,0.1)",
        ),
        bgcolor="rgba(0,0,0,0)",
    ),
    showlegend=True,
    legend=dict(x=0.85, y=1.15),
    title=dict(
        text="Embedding Failure Severity by Category",
        font=dict(size=16),
        x=0.5,
    ),
    width=600,
    height=500,
    margin=dict(t=80, b=40, l=80, r=80),
)

fig.show()

# Print the raw severity values
print("\nFailure Severity Scores (0 = no failure, 10 = completely broken):")
for cat, sev in zip(categories, severities):
    bar = "#" * int(sev) + "." * (10 - int(sev))
    print(f"  {cat:<16s} [{bar}] {sev:.1f}/10")

---

## The Takeaway

Embeddings are the foundation of semantic search, but they have real blind spots:

| Failure Mode | What Happens | Risk Level |
|-------------|-------------|------------|
| **Polysemy** | Same word in different contexts gets conflated | Medium -- context usually disambiguates |
| **Negation** | "is" and "is not" produce nearly identical vectors | **High** -- opposite meanings, same score |
| **Word Order** | Swapped roles look the same to the model | Medium -- depends on domain |
| **Length Mismatch** | Short queries vs long documents produce skewed scores | **High** -- affects every real retrieval system |
| **Register** | Formal and informal language for the same concept diverge | Medium -- depends on user population |

These failures are exactly why **naive RAG is not enough**. In Session 3.1, you will see these problems surface when building a retrieval pipeline -- the same patterns that scored misleadingly here will cause your pipeline to return wrong documents.

The solutions we will build in Sessions 3.1 and 3.2:
- **Chunking** (Session 2.2) addresses length mismatch by breaking documents into smaller, more matchable pieces
- **Metadata filtering** (Session 3.2) addresses polysemy and register by narrowing the search space before similarity comparison
- **Evaluation frameworks** let you measure how often these failures actually hurt your specific use case

The skill is not avoiding embeddings -- they are the best tool we have for semantic search. The skill is knowing *where they break* so you can engineer around it.